# 03. H1 прогон модели

Запускаем одиночные запросы и батчи для одной модели. Для каждой модели — отдельный запуск с другим `MODEL_KEY`.

**Порядок:**
1. `MODEL_KEY = "mock"`, `PILOT = True` — бесплатная проверка, что код работает.
2. Реальная модель, `PILOT = True` (20 целевых, ≈ 320 запросов) — смотрим долю нераспознанных ответов, отказы, logprobs, токены.
3. Та же модель, `PILOT = False` полный прогон (1 100 одиночных + 4 500 батчей).

Все ответы кэшируются в `data/cache/<модель>.jsonl`

In [1]:
MODEL_KEY = "gigachat_lite" 
LANG = "en"
PILOT = False           # True 20 целевых False все 300
N_PILOT = 20

In [2]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
DATA_PROC = PROJECT_DIR / "data" / "processed"
DATA_RES = PROJECT_DIR / "data" / "results"
DATA_RES.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 120)

from src.config import MODELS
from src.llm import ask_many
from src.prompts import SYSTEM, build_user_prompt, parse_labels, p_toxic_from_logprobs

targets = pd.read_parquet(DATA_PROC / "targets.parquet")
neighbors = pd.read_parquet(DATA_PROC / "neighbors_pool.parquet")
plan = pd.read_parquet(DATA_PROC / "h1_plan.parquet")
text = pd.concat([targets, neighbors]).set_index("id")["text"]
system = SYSTEM[LANG]

if PILOT:
    pilot_ids = (targets.groupby("target_group", group_keys=False)
                        .apply(lambda g: g.sample(N_PILOT // 2, random_state=0))["id"])
    plan = plan[plan["target_id"].isin(pilot_ids)]
    single_ids = list(pilot_ids) + sorted({i for ids in plan["item_ids"] for i in ids} - set(pilot_ids))
else:
    single_ids = list(targets["id"]) + list(neighbors["id"])

suffix = f"{MODEL_KEY}_{LANG}" + ("_pilot" if PILOT else "")
print(MODELS[MODEL_KEY])
print(f"одиночных: {len(single_ids)}, батчей: {len(plan)}")

{'provider': 'gigachat', 'model': 'GigaChat-2', 'logprobs': False, 'workers': 1}
одиночных: 1100, батчей: 4500


## Шаг 1. Одиночные запросы
Прогоняем каждый текст отдельно, с которым потом будем сравнивать ответ в батче.

In [3]:
def to_rows(resp, n, extra):
    labels, ok = parse_labels(resp["text"], n)
    probs = p_toxic_from_logprobs(resp["tokens"], n) if resp.get("tokens") else [None] * n
    u = resp.get("usage") or {}
    return dict(extra, labels=labels, probs=probs, parse_ok=ok, finish_reason=resp.get("finish_reason"),
                error=resp.get("error"), in_tokens=u.get("input"), out_tokens=u.get("output"), raw=resp["text"])

jobs = [dict(id=int(i), system=system, user=build_user_prompt([text[i]]), max_tokens=16) for i in single_ids]
resp = ask_many(MODEL_KEY, jobs, desc="одиночные")

singles = pd.DataFrame([to_rows(resp[i], 1, dict(id=i)) for i in resp])
singles["y_single"] = singles["labels"].str[0]
singles["p_single"] = singles["probs"].str[0]
singles = singles.drop(columns=["labels", "probs"])
gold = pd.concat([targets, neighbors]).set_index("id")["label"]
singles["gold"] = singles["id"].map(gold)
singles.head()

одиночные:   0%|          | 0/1100 [00:00<?, ?it/s]

,id,parse_ok,finish_reason,error,in_tokens,out_tokens,raw,y_single,p_single,gold
0,5903407,True,stop,None,170,7,1: TOXIC,1,None,1
1,6112593,True,stop,None,83,7,1: TOXIC,1,None,1
2,881045,True,stop,None,79,7,1: TOXIC,1,None,1
3,706298,True,stop,None,184,7,1: TOXIC,1,None,1
4,5670395,True,stop,None,51,7,1: TOXIC,1,None,1


In [4]:
print("ошибки API:            ", singles["error"].notna().sum())
print("ответ не распознан:    ", (~singles["parse_ok"]).sum())
print("отказы (finish_reason):", singles["finish_reason"].value_counts().to_dict())
print("есть logprobs:         ", singles["p_single"].notna().mean().round(3))
ok = singles.dropna(subset=["y_single"])
print("точность vs разметка:  ", (ok["y_single"] == ok["gold"]).mean().round(3))
print("токенов на запрос (вход / выход):", singles["in_tokens"].mean().round(0), "/", singles["out_tokens"].mean().round(1))
singles.loc[~singles["parse_ok"], ["id", "raw", "finish_reason", "error"]].head()

ошибки API:             0
ответ не распознан:     0
отказы (finish_reason): {'stop': 1100}
есть logprobs:          0.0
точность vs разметка:   0.773
токенов на запрос (вход / выход): 72.0 / 7.5


,id,raw,finish_reason,error


## Шаг 2. Батчи
Целевой + 8 соседей; модель отвечает метками всех 9 текстов, берём метку (и вероятность) целевого по его позиции.

In [5]:
jobs = [dict(id=r.batch_id, system=system, user=build_user_prompt([text[i] for i in r.item_ids]), max_tokens=120)
        for r in plan.itertuples()]
resp = ask_many(MODEL_KEY, jobs, desc="батчи")

rows = [to_rows(resp[r.batch_id], len(r.item_ids), dict(batch_id=r.batch_id)) for r in plan.itertuples()]
batches = plan.merge(pd.DataFrame(rows), on="batch_id")
batches["y_batch"] = [lab[p - 1] for lab, p in zip(batches["labels"], batches["target_pos"])]
batches["p_batch"] = [pr[p - 1] for pr, p in zip(batches["probs"], batches["target_pos"])]
batches[["batch_id", "prop_toxic", "target_pos", "y_batch", "p_batch", "parse_ok", "finish_reason"]].head()

батчи:   0%|          | 0/4500 [00:00<?, ?it/s]

,batch_id,prop_toxic,target_pos,y_batch,p_batch,parse_ok,finish_reason
0,h1_5903407_k0_r0,0.00,1,1.0,None,True,stop
1,h1_5903407_k2_r0,0.25,1,1.0,None,True,stop
2,h1_5903407_k4_r0,0.50,1,1.0,None,True,stop
3,h1_5903407_k6_r0,0.75,1,1.0,None,True,stop
4,h1_5903407_k8_r0,1.00,1,1.0,None,True,stop


In [6]:
print("ошибки API:                 ", batches["error"].notna().sum())
print("ответ не распознан целиком: ", (~batches["parse_ok"]).mean().round(3))
print("нет метки целевого:         ", batches["y_batch"].isna().mean().round(3))
print("токенов на батч (вход / выход):", batches["in_tokens"].mean().round(0), "/", batches["out_tokens"].mean().round(1))
print("\nДоля нераспознанных по доле токсичных соседей (не должна сильно расти):")
print(batches.groupby("prop_toxic")["y_batch"].apply(lambda s: s.isna().mean()).round(3))

ошибки API:                  0
ответ не распознан целиком:  0.001
нет метки целевого:          0.0
токенов на батч (вход / выход): 426.0 / 59.5

Доля нераспознанных по доле токсичных соседей (не должна сильно расти):
prop_toxic
0.00    0.000
0.25    0.000
0.50    0.000
0.75    0.001
1.00    0.001
Name: y_batch, dtype: float64


In [7]:
singles.to_parquet(DATA_RES / f"h1_singles_{suffix}.parquet", index=False)
batches.drop(columns=["raw"]).to_parquet(DATA_RES / f"h1_batches_{suffix}.parquet", index=False)
print("Сохранено:", f"h1_singles_{suffix}.parquet", f"h1_batches_{suffix}.parquet")

# Быстрый взгляд: доля ответов TOXIC у целевого по доле токсичных соседей
quick = batches.merge(singles[["id", "y_single"]], left_on="target_id", right_on="id")
quick.groupby("prop_toxic")[["y_single", "y_batch"]].mean().round(3)

Сохранено: h1_singles_gigachat_lite_en.parquet h1_batches_gigachat_lite_en.parquet


,y_single,y_batch
prop_toxic,,
0.00,0.59,0.630
0.25,0.59,0.607
0.50,0.59,0.570
0.75,0.59,0.558
1.00,0.59,0.541
